# convolution classifier convolution with explicit im2col

Convolution is implemented with explicit image-to-column windows and matrix multiplication. Column-to-image accumulates overlapping contributions. Pooling uses explicit windows. Reference tests check values and all input/weight/bias gradients. The 29 ordered tabular features are standardized and zero-padded into a 7×7 grid; this is an artificial feature arrangement, not a natural-image benchmark.


In [ ]:
import json,sys
from pathlib import Path
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'project.json').exists())
sys.path.insert(0,str(ROOT))
import numpy as np
import models


## Implementation check

This mathematical smoke check is separate from measured classifier accuracy.


In [ ]:
import torch
from experiment_core import as_tensor,torch_factories,SPECS
kind='convolution'
factory=next(iter(torch_factories(kind).values()))[0]
model=factory().eval()
with torch.no_grad():
    probs=model(as_tensor(np.zeros((3,SPECS[kind][0])),kind)).softmax(dim=1)
assert torch.isfinite(probs).all() and torch.allclose(probs.sum(dim=1),torch.ones(3))


## Experiment and held-out results

Retrain using the README command and local project CSV. The results below are from the fresh complete run; official unlabelled test accuracy is unavailable.


In [ ]:
metrics=json.loads((ROOT/'metrics.json').read_text())
assert all(m['checkpoint_reload_identical'] for m in metrics['models'].values())
[(name,m['test']['accuracy'],m['test']['macro_f1']) for name,m in metrics['models'].items()]


One fixed stratified split measures experimental performance on this dataset. It does not certify generalization outside the dataset or deployment reliability.
